In [1]:
!apt-get update -qq
!apt-get install -y ns2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libotcl1 libpcap0.8t64 libtclcl1
Suggested packages:
  gnuplot
The following NEW packages will be installed:
  libotcl1 libpcap0.8t64 libtclcl1 ns2
0 upgraded, 4 newly installed, 0 to remove and 68 not upgraded.
Need to get 2,810 kB of archives.
After this operation, 15.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libpcap0.8t64 amd64 1.10.4-4.1ubuntu3.1 [151 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libotcl1 amd64 1.14+dfsg-8 [24.2 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 libtclcl1 amd64 1.20-10 [65.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu n

In [2]:
%%writefile exp5.tcl

set ns [new Simulator]

# Enable Distance Vector routing
$ns rtproto DV

set tr [open out.tr w]
set nf [open out.nam w]

$ns trace-all $tr
$ns namtrace-all $nf

# Create four nodes
for {set i 0} {$i < 4} {incr i} {
    set n($i) [$ns node]
}

# Create ring topology
for {set i 0} {$i < 4} {incr i} {
    $ns duplex-link $n($i) $n([expr ($i+1)%4]) 1Mb 10ms DropTail
}

# UDP connection from node 0 to node 2
set udp [new Agent/UDP]
set null [new Agent/Null]

$ns attach-agent $n(0) $udp
$ns attach-agent $n(2) $null
$ns connect $udp $null

# CBR traffic
set cbr [new Application/Traffic/CBR]
$cbr attach-agent $udp
$cbr set packetSize_ 512
$cbr set interval_ 0.05

# Start traffic
$ns at 0.5 "$cbr start"

# Link failure and recovery
$ns rtmodel-at 2.0 down $n(1) $n(2)
$ns rtmodel-at 3.0 up $n(1) $n(2)

# Stop traffic and finish
$ns at 4.5 "$cbr stop"
$ns at 5.0 "finish"

proc finish {} {
    global ns tr nf
    $ns flush-trace
    close $tr
    close $nf
    exit 0
}

$ns run

Writing exp5.tcl


In [3]:
!ns exp5.tcl

In [4]:
!ls

exp5.tcl  out.nam  out.tr  sample_data


In [5]:
!head out.tr

+ 0.00017 0 1 rtProtoDV 4 ------- 0 0.2 1.1 -1 0
- 0.00017 0 1 rtProtoDV 4 ------- 0 0.2 1.1 -1 0
+ 0.00017 0 3 rtProtoDV 4 ------- 0 0.2 3.1 -1 1
- 0.00017 0 3 rtProtoDV 4 ------- 0 0.2 3.1 -1 1
+ 0.007102 2 1 rtProtoDV 4 ------- 0 2.2 1.1 -1 2
- 0.007102 2 1 rtProtoDV 4 ------- 0 2.2 1.1 -1 2
+ 0.007102 2 3 rtProtoDV 4 ------- 0 2.2 3.1 -1 3
- 0.007102 2 3 rtProtoDV 4 ------- 0 2.2 3.1 -1 3
r 0.010202 0 1 rtProtoDV 4 ------- 0 0.2 1.1 -1 0
+ 0.010202 1 0 rtProtoDV 4 ------- 0 1.1 0.2 -1 4


In [6]:
!awk '$1=="d" {n++} END {print "Dropped packets:",n+0}' out.tr

Dropped packets: 0


In [7]:
!awk '$1=="r" {n++} END {print "Received packets:",n+0}' out.tr

Received packets: 200
